# Week 5 — Wednesday: PCA — Seeing High-Dimensional Data

**DATA 202 · Calvin University**

Monday: **2 numbers** per point → one scatter plot showed everything. A handwritten digit is **64 numbers**.

> 🎯 **How do you look at data you can't plot?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~8 min | 1 · Digits: clustering 64 numbers per image |
| ~7 min | 2 · Choosing a view: what PCA does |
| ~7 min | 3 · Scaling: one ruler for every column |
| ~10 min | 4 · PCA on digits: 64 → 2 (SLO 05C) |
| ~5 min | 5 · How many components? |
| ~3 min | Wrap-up |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk

---
## 1 · Digits: 64 Numbers per Image · ~8 min

**digits** — 1,797 handwritten digits, 8×8 pixels

* Each image → **64 numbers** (pixel darkness, 0–16)
* 10 classes, 0–9 — *we'll ignore the labels for now*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()
X = digits.data              # one row per image, one column per pixel
print(X.shape)               # (1797, 64)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(10, 4))
for image, ax in zip(digits.images, axes.flat):
    ax.imshow(image, cmap="gray_r")
    ax.axis("off")
plt.show()

Monday's recipe doesn't care how many columns there are: "nearest centroid" works on 64 numbers just like on 2.

> 🎯 **Predict First:** Cluster the digits into 10 groups. Will the clusters line up with the digits 0–9?

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
kmeans.fit(X)
labels = kmeans.labels_

In [ ]:
fig, axes = plt.subplots(nrows=10, ncols=9, figsize=(6.5, 7.5))
for c in range(10):
    axes[c, 0].imshow(kmeans.cluster_centers_[c].reshape(8, 8), cmap="gray_r")   # the cluster's average image
    for j, image in enumerate(X[labels == c][:8], start=1):                       # 8 of its members
        axes[c, j].imshow(image.reshape(8, 8), cmap="gray_r")
    axes[c, 0].set_ylabel(f"cluster {c}", rotation=0, ha="right", va="center")
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_title("average")
plt.show()

> 📄 **Handout 1 — What's in each cluster?** Write the digit each row mostly holds; circle the images that don't belong. 🗣️ Compare with a neighbor.

<details><summary>Answer</summary>

* Clusters 0 → 9 mostly hold: **2, 0, 1, 8, 7, 6, 3, 5, 9, 4**
* Most mixed: **cluster 3** — 8s with 1s and 2s (overall about 100 ones *and* 100 eights) — and **cluster 2** — 1s with a 7 and a 2
* No labels were used — yet most clusters *are* one digit
</details>

---
## 2 · Choosing a View: What PCA Does · ~7 min

64 numbers = 64 axes. We can only *look* at 2. A plot is a **view** — a shadow of the data on a flat wall.

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/05/images/pca_perspective.png" width="420" alt="A cylinder lit from two sides casts two different shadows on two walls, a diamond and an ellipse. Caption: How something appears is always a matter of perspective.">

> "I believe in Christianity as I believe that the sun has risen: not only because I see it, but because by it I see everything else." — C. S. Lewis

Some perspectives show far more than others.

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/05/images/pca_rotating_rat.gif" width="300" alt="A 3-D model of a rat rotating, so that its 2-D shadow changes shape">

**PCA** rotates the data to find the most revealing views — no labels needed:

* **PC1** — the direction where the points are most **spread out** (most variance)
* **PC2** — the most spread left over, at a right angle to PC1
* … one per column. Keep the first few → **fewer numbers, most of the information**

*(In math terms: these directions are the eigenvectors of the data's covariance matrix; the spread along each is its eigenvalue.)*

---
## 3 · Scaling: One Ruler for Every Column · ~7 min

All 64 pixels share one ruler: darkness from 0 to 16. PCA chases **variance** — and variance depends on **units**.

> 🎯 **Predict First:** Suppose a scanner stored pixel 42 on a 0–1,600 scale instead of 0–16 — the same images, that one column just ×100. What happens to PC1?

In [ ]:
X_units = X.copy()
X_units[:, 42] = X_units[:, 42] * 100          # same images: pixel 42 now on a 0–1,600 scale

share = PCA().fit(X_units).explained_variance_ratio_[:2] * 100      # % of the variance, PC1 and PC2
print(pd.Series(share, index=["PC1", "PC2"]).round(1))

**PC1 keeps 99.7% of the variance — and it is just pixel 42.** Bigger numbers → bigger variance → PCA decides that one pixel is the whole story.

**The fix — `StandardScaler`:** for each column, **z = (value − column mean) ÷ column standard deviation** → every column gets mean 0 and standard deviation 1.

> 📄 **Handout 2 — One ruler.** Compute pixel 42's z-score for the first image, in both units. Then check below.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X_units)
print("first image, pixel 42, as a z-score:", X_scaled[0, 42].round(2))   # check your handout

share = PCA().fit(X_scaled).explained_variance_ratio_[:2] * 100
print(pd.Series(share, index=["PC1", "PC2"]).round(1))

**0.63 in both units** — a z-score counts standard deviations, so the units cancel out. After scaling, no single pixel dominates: PC1 keeps 12.0%, PC2 9.6%.

| Columns are… | Do this |
|---|---|
| in different units (grams vs. mm, $ vs. %, our ×100 pixel) | **Scale first** |
| already on one ruler (the real 64 pixels: 0–16) | **Skip it** — so from here on we use the original digits, no scaler |

---
## 4 · PCA on Digits: 64 → 2 (SLO 05C) · ~10 min

Goal: **look at** the 10 clusters from section 1.

* `PCA(n_components=2)` → keep the 2 best views
* `fit_transform` → each image becomes 2 numbers

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)                    # 1,797 × 64  →  1,797 × 2

print(X_2d.shape)
print(pca.explained_variance_ratio_.round(3))

In [ ]:
from plotly.subplots import make_subplots

jitter = np.random.default_rng(0).uniform(-0.3, 0.3, size=(len(X), 2))   # pixels are whole numbers: spread overlapping dots
palette = px.colors.qualitative.Plotly                                     # 10 colors: one per cluster
fig = make_subplots(rows=1, cols=2, subplot_titles=["Before: pixel 42 vs. pixel 21", "After: PC1 vs. PC2"])
for c in range(10):
    in_c = labels == c
    fig.add_scatter(x=X[in_c, 42] + jitter[in_c, 0], y=X[in_c, 21] + jitter[in_c, 1], mode="markers",
                    marker=dict(size=4, color=palette[c], opacity=0.6),
                    name=f"cluster {c}", legendgroup=str(c), row=1, col=1)
    fig.add_scatter(x=X_2d[in_c, 0], y=X_2d[in_c, 1], mode="markers",
                    marker=dict(size=4, color=palette[c], opacity=0.6),
                    name=f"cluster {c}", legendgroup=str(c), showlegend=False, row=1, col=2)
fig.update_layout(height=450, title="Digits k-means clusters: before and after PCA")
fig.show()

pixel_share = X[:, [42, 21]].var(axis=0).sum() / X.var(axis=0).sum()
print(f"variance kept — 2 pixels: {pixel_share:.1%}   PC1 + PC2: {pca.explained_variance_ratio_.sum():.1%}")

* **Before:** 2 of the 64 pixels keep **6.8%** of the variance — clusters pile up
* **After:** PC1 + PC2 keep **28.5%** — most clusters get their own region
* Still a *shadow* of the 64-D data — just a much better-chosen one

> 📄 **Handout 3 — Two views.** Circle the 0s in both views; name two digits that still overlap after PCA. 🗣️ Compare with a neighbor.

<details><summary>Answer</summary>

* The **0s** (and the 6s and 4s) form their own group only in the **PCA** view
* Still overlapping after PCA: **5 & 8**, **1 & 7**, **3 & 9**
* Overlap in a shadow can vanish in 64-D: this view keeps 28.5% of the variance, not all of it
</details>

---
## 5 · How Many Components? · ~5 min

2 components keep 28.5%. What if we keep more?

> 📄 **Handout 4:** mark where the curve reaches 80% and 90%.

In [ ]:
pca_full = PCA(random_state=42).fit(X)                  # keep all 64 components
cumulative = np.cumsum(pca_full.explained_variance_ratio_)

fig = px.line(x=range(1, 65), y=cumulative,
              labels={"x": "Number of components", "y": "Cumulative variance kept"},
              title="Digits: variance kept by the first n components")
fig.add_hline(y=0.8, line_dash="dot")
fig.add_hline(y=0.9, line_dash="dot")
fig.show()

n_80 = int(np.argmax(cumulative >= 0.8)) + 1            # first component where the curve reaches 80%
n_90 = int(np.argmax(cumulative >= 0.9)) + 1
print("components for 80%:", n_80, "| for 90%:", n_90)

<details><summary>Answer</summary>

**13 components keep 80%; 21 keep 90%** — 21 of the 64 numbers hold 90% of the variation.
</details>

---
## Wrap-up · ~3 min

* **k-means** works on 64 numbers just like on 2
* **PCA** finds the best views: PC1 keeps the most variance, PC2 the next…
* **Scale first** when columns are in different units
* A 2-D plot is a **shadow** — always ask how much variance it keeps

> 💬 **Question:** Today we clustered first and used PCA only to *look*. Why might someone reduce *first*, then cluster?

<details><summary>Answer</summary>

* Distances get noisy with many columns — and fewer columns run faster
* But PCA keeps the most-*spread* directions, not necessarily the ones that *separate* groups
* Common compromise: keep enough components for most of the variance (like 21 for 90%), cluster on those, and use 2 only to plot
</details>

---
### ⭐ Optional — Denoising a Signal With PCA (if there's time)

* Close to a technique Prof. Santos used on real neuron recordings (published research)
* Stack short overlapping windows of a noisy signal into a matrix → rebuild it from only the first principal component

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Synthetic neuron membrane potential with chaotic low-frequency noise
np.random.seed(42)
time = np.linspace(0, 1, 1000)  # 1 second at 1000 Hz
membrane_potential = -70 + 10 * np.sin(2 * np.pi * 50 * time)
baseline_shifts = np.piecewise(time, [time < 0.3, (time >= 0.3) & (time < 0.6), time >= 0.6], [-2, 5, -3])
chaotic_noise = baseline_shifts + 1.5 * np.sin(2 * np.pi * 0.5 * time)
chaotic_signal_with_noise = membrane_potential + chaotic_noise

def create_signal_matrix(signal, lag):
    rows = len(signal) - lag + 1
    return np.array([signal[i:i + lag] for i in range(rows)])

def pca_denoise(signal, lag, n_components=1):
    M = create_signal_matrix(signal, lag)
    pca = PCA(n_components=n_components)
    M_reduced = pca.fit_transform(M)
    M_reconstructed = pca.inverse_transform(M_reduced)
    return np.mean(M_reconstructed, axis=1)  # back to 1D by averaging the diagonals

denoised_pca_signal = pca_denoise(chaotic_signal_with_noise, lag=20, n_components=1)

plt.figure(figsize=(15, 6))
plt.plot(time, chaotic_signal_with_noise, label="Recorded Signal", color='orange')
plt.plot(time, membrane_potential, label="True Membrane Potential", color='green', linestyle='dashed')
plt.xlabel("Time (s)")
plt.ylabel("Membrane Potential (mV)")
plt.legend()
plt.title("Denoising of Membrane Potential Signal Using PCA")
plt.show()